# Multinomial Naive Bayes

In [1]:
# Basic Imports
import sys
import pandas as pd
import time
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Importing shared functions
from src.preprocessing import preprocess
from src.vectorization import vectorize
from src.evaluation import evaluate
from src.experiments import run_experiment

# Model-specific imports
from sklearn.naive_bayes import MultinomialNB

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aparajaya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
df_raw = pd.read_csv("../data/Dataset.csv")

## Model 1 

Model - Multinomial Naive Bayes     
Vectorization - TF-IDF with N-grams             

### Baseline:

Features -      
* TF-IDF 
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time : 45.0538s   
* Training Time : 0.0758 s              

Metrics -           
* Accuracy:  0.91989
* Precision:  0.92579
* Recall:  0.91658
* F1:  0.91986
* Confusion Matrix:     
  [10756    893         -> False Positives      
   1014     11142]      -> False Negatives      

In [3]:
# Creating features
start_time = time.perf_counter()

df = preprocess(df_raw, check=True)
X_train, X_test, Y_train, Y_test, feature_names, total_features = vectorize(df, method = "tfidf")

end_time = time.perf_counter()
creation_time = end_time - start_time

print(f"\nFeatures created in {creation_time:.4f} seconds.")
print("Total features = ",total_features)

No of empty title rows:  563
No of empty text rows:  786
No of rows with empty title AND text:  1

Features created in 52.8526 seconds.
Total features =  477076


In [4]:
# Training model
start_time = time.perf_counter()

model = MultinomialNB().fit(X_train, Y_train)
Y_pred = model.predict(X_test)

end_time = time.perf_counter()
training_time = end_time - start_time

print(f"Training completed in {training_time:.4f} seconds.")

Training completed in 0.0932 seconds.


In [5]:
# Evaluation
accuracy, precision, recall, f1, cm, top_negative_features, top_positive_features = evaluate(Y_test, Y_pred, model, feature_names)

Accuracy:  0.9198907792480572
Precision:  0.9257997507270461
Recall:  0.9165844027640672
F1:  0.9198697106834397

Confusion Matrix:
 [[10756   893]
 [ 1014 11142]]

Top 5 Negative Features:
                   Feature  Importance
454520  washington reuters   -5.005159
353464          reuters us   -4.388347
15915           york times   -4.381987
4836               factbox   -4.344310
357062            rohingya   -3.874943

Top 5 Positive Features:
             Feature  Importance
205953     image via    4.783528
83597   century wire    3.967394
15822            wow    3.958042
1648     boiler room    3.825864
1647          boiler    3.825864


In [6]:
# Storing results
results = []

results.append({
    "Experiment": "Baseline",
    "Vectorizer": "tfidf",
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Features": total_features,
    "Feature Time (s)": creation_time,
    "Training Time (s)": training_time
})

## Experiments

In [7]:
model = MultinomialNB()

### Experiment 1 (Unigrams Only)

In [8]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 19.0473 seconds.
Total features =  70042
Training completed in 0.0401 seconds.
Accuracy:  0.8989287964713295
Precision:  0.9009705543674946
Recall:  0.9011187890753537
F1:  0.898882566328855

Confusion Matrix:
 [[10445  1204]
 [ 1202 10954]]

Top 5 Negative Features:
        Feature  Importance
3145    factbox   -4.551333
54977  rohingya   -4.164879
11940     _____   -3.973370
7216   rohingya   -3.922725
9442        000   -3.887749

Top 5 Positive Features:
         Feature  Importance
9342         wow    4.059732
1103      boiler    3.946714
10630     21wire    3.806087
4014   hilarious    3.805463
12394        acr    3.788624


### Experiment 2 (Vary Alpha)

In [9]:
alpha_variation = [0.1, 0.5, 2.0] # Default = 1.0
for ALPHA in alpha_variation:
    model_var = model = MultinomialNB(alpha=ALPHA)
    results.append(
        run_experiment(
            df_raw,
            model=model_var,
            vectorizer="tfidf",
            experiment_name=f"Alpha={ALPHA}",
            ngramRange=(1,2)
        )
    )


Features created in 53.4114 seconds.
Total features =  477076
Training completed in 0.0762 seconds.
Accuracy:  0.9303507666456626
Precision:  0.9284198498204375
Recall:  0.9357518920697598
F1:  0.9303060475154847

Confusion Matrix:
 [[10772   877]
 [  781 11375]]

Top 5 Negative Features:
                   Feature  Importance
454520  washington reuters   -6.946544
4836               factbox   -6.621205
11557             rohingya   -6.063213
2222               catalan   -6.015038
357062            rohingya   -5.886638

Top 5 Positive Features:
               Feature  Importance
205953       image via    7.092554
83597     century wire    6.266656
1648       boiler room    6.122449
1647            boiler    6.122449
370683  screen capture    5.928980

Features created in 50.8923 seconds.
Total features =  477076
Training completed in 0.0894 seconds.
Accuracy:  0.9234614576769586
Precision:  0.9257580751483191
Recall:  0.9242349457058243
F1:  0.9234294109604269

Confusion Matrix:
 [[107

### Experiment 3 (Remove Source Identifiers)

In [10]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="tfidf",
        experiment_name="Removed source names",
        remove_sourcewords=True
    )
)


Features created in 57.1788 seconds.
Total features =  475888
Training completed in 0.0762 seconds.
Accuracy:  0.8933837429111531
Precision:  0.8893927125506073
Recall:  0.9035867061533399
F1:  0.8932912092250346

Confusion Matrix:
 [[10283  1366]
 [ 1172 10984]]

Top 5 Negative Features:
         Feature  Importance
4766     factbox   -3.672491
15784        000   -3.256474
355885  rohingya   -3.229246
273812   myanmar   -3.210046
11405   rohingya   -3.131942

Top 5 Positive Features:
             Feature  Importance
205465     image via    4.092474
14943          video    3.566337
1755        breaking    3.384972
15625            wow    3.374541
83354   century wire    3.286732


## Model 2

Model - Naive Bayes     
Vectorization - CountVectorizer 

### Baseline:
Features -      

* CountVectorizer  
    - min_df = 5
    - max_features=50000 for title, 500000 for text
* N-grams
    - Uni + Bigrams
* Text and title processed separately
* Stopwords removed
* Total features = 477k      

Time - 
* Feature Creation time :  48.4s 
* Training Time :  0.0781s          

Metrics -           
* Accuracy:  0.92774
* Precision:  0.93856
* Recall:  0.91864
* F1:  0.92773
* Confusion Matrix:         
    [10918   731       -> False Positives            
     989     11167]     -> False Negatives         

In [11]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="Baseline",
        ngramRange=(1,2)
    )
)


Features created in 48.4050 seconds.
Total features =  477076
Training completed in 0.0781 seconds.
Accuracy:  0.9277462717916404
Precision:  0.9385611027063372
Recall:  0.9186410003290556
F1:  0.9277383655496478

Confusion Matrix:
 [[10918   731]
 [  989 11167]]

Top 5 Negative Features:
                   Feature  Importance
454520  washington reuters   -7.177839
24452                _____   -5.866042
353326   reuters president   -5.758709
357062            rohingya   -5.720616
335990             rakhine   -5.552696

Top 5 Positive Features:
               Feature  Importance
205953       image via    7.772820
83597     century wire    6.615539
476966             что    6.397997
476127              не    6.191932
370683  screen capture    6.183891


## Experiments

### Experiment 1 (Unigrams Only)

In [12]:
results.append(
    run_experiment(
        df_raw,
        model=model,
        vectorizer="count",
        experiment_name="Unigrams Only",
        ngramRange=(1,1)
    )
)


Features created in 16.6591 seconds.
Total features =  70042
Training completed in 0.0369 seconds.
Accuracy:  0.9083805923125394
Precision:  0.9136601144563324
Recall:  0.9062191510365252
F1:  0.9083536708264988

Confusion Matrix:
 [[10608  1041]
 [ 1140 11016]]

Top 5 Negative Features:
          Feature  Importance
11940       _____   -5.868847
54977    rohingya   -5.723421
52361     rakhine   -5.555501
51712  puigdemont   -5.358866
30014         fdp   -4.843105

Top 5 Positive Features:
                   Feature  Importance
69963                  что    6.395192
69274                   не    6.189128
10630               21wire    6.126570
10506  2017realdonaldtrump    6.091870
69982                  это    5.759928


### Experiment 2 (Vary Alpha)

In [13]:
model_var = MultinomialNB(alpha=0.1)
results.append(
    run_experiment(
        df_raw,
        model=model_var,
        vectorizer="count",
        experiment_name="Alpha=0.1",
        ngramRange=(1,2)
    )
)


Features created in 48.6291 seconds.
Total features =  477076
Training completed in 0.0875 seconds.
Accuracy:  0.9394664986347406
Precision:  0.9478391707765611
Recall:  0.9327903915761764
F1:  0.9394559646110978

Confusion Matrix:
 [[11025   624]
 [  817 11339]]

Top 5 Negative Features:
                  Feature  Importance
353326  reuters president   -8.742286
335990            rakhine   -8.535658
310304        pic twitter   -8.444703
245260     london reuters   -8.344639
331812         puigdemont   -8.338305

Top 5 Positive Features:
               Feature  Importance
205953       image via   10.777577
83597     century wire    9.619319
476966             что    9.401431
476127              не    9.194960
370683  screen capture    9.186901


In [14]:
comparison_df = pd.DataFrame(results)
comparison_df.sort_values("Accuracy", ascending=False)

,Experiment,Vectorizer,Accuracy,Precision,Recall,F1,Features,Feature Time (s),Training Time (s)
8,Alpha=0.1,count,0.939466,0.947839,0.932790,0.939456,477076,48.629064,0.087477
2,Alpha=0.1,tfidf,0.930351,0.928420,0.935752,0.930306,477076,53.411381,0.076209
6,Baseline,count,0.927746,0.938561,0.918641,0.927738,477076,48.405010,0.078053
3,Alpha=0.5,tfidf,0.923461,0.925758,0.924235,0.923429,477076,50.892290,0.089431
0,Baseline,tfidf,0.919891,0.925800,0.916584,0.919870,477076,52.852568,0.093154
4,Alpha=2.0,tfidf,0.915438,0.925283,0.907700,0.915427,477076,51.821055,0.073422
7,Unigrams Only,count,0.908381,0.913660,0.906219,0.908354,70042,16.659109,0.036901
1,Unigrams Only,tfidf,0.898929,0.900971,0.901119,0.898883,70042,19.047289,0.040107
5,Removed source names,tfidf,0.893384,0.889393,0.903587,0.893291,475888,57.178812,0.076218


In [16]:
comparison_df.to_csv(
    "../results/naive_bayes.csv",
    index=False
)

## Model Analysis and Experimental Findings

### Baseline Performance

Multinomial Naive Bayes with TF-IDF vectorization achieved:

- Accuracy: 91.99%
- Precision: 92.58%
- Recall: 91.66%
- F1 Score: 91.99%

Although performance was substantially lower than Logistic Regression, training was extremely fast (<0.1 seconds).

---

### Vectorization Analysis

CountVectorizer consistently outperformed TF-IDF.

| Configuration | Accuracy |
|---------------|----------|
| TF-IDF        | 91.99%   |
| Count         | 92.77%   |
| Count + α=0.1 | 93.95%   |

This behavior is expected because Multinomial Naive Bayes models word occurrence counts directly.

---

### Effect of Smoothing

Reducing the smoothing parameter improved performance.

| α | Accuracy |
|---|----------|
| 0.1 | 93.04% |
| 0.5 | 92.35% |
| 1.0 | 91.99% |
| 2.0 | 91.54% |

Lower smoothing preserves the influence of informative features, while excessive smoothing reduces their discriminative power.

---

### N-Gram Analysis

Removing bigrams reduced performance for both vectorizers.

| Configuration | Accuracy |
|---------------|----------|
| TF-IDF + Unigrams | 89.89% |
| Count + Unigrams | 90.84% |

Bigrams provide useful contextual information and partially compensate for Naive Bayes' independence assumption.

---

### Impact of Source Identifiers

Removing publisher names and source phrases caused a significant performance drop.

- Accuracy: 91.99% → 89.34%

This suggests that Naive Bayes relies heavily on highly discriminative tokens such as publisher names and source identifiers.

### Key Conclusions

- Logistic Regression substantially outperformed Naive Bayes.
- CountVectorizer performed better than TF-IDF.
- Lower smoothing values improved performance.
- Bigrams provided meaningful gains.
- Naive Bayes relied more heavily on source identifiers than Logistic Regression.
- Training time was extremely low, but vectorization remained the primary computational bottleneck.